# Lab 1 — Perceptrons and Multi-Layer Perceptrons

**Module:** 7144COMP — Deep Learning Concepts and Techniques  
**Week:** 1  
**Estimated time:** 90 minutes

---

## Learning outcomes

By the end of this lab you should be able to:

1. Describe the components of a perceptron (inputs, weights, bias, activation, output) and write the update rule from memory.
2. Implement a perceptron *from scratch* in NumPy and explain what happens during each training epoch.
3. Explain why a single perceptron **cannot** solve the XOR problem, and why a multi-layer perceptron (MLP) can.
4. Implement forward and backward propagation for a small MLP by hand.
5. Reproduce the same result using scikit-learn's high-level `Perceptron` and `MLPClassifier`, and justify when you would use a from-scratch implementation versus a library.

## Prerequisites

- A working Python 3 environment (this is provided automatically by the module's Docker container).
- Comfort with NumPy array operations and basic plotting with Matplotlib.
- Lecture 1: *Foundations of Neural Networks* — in particular the slides on linear classifiers, the perceptron learning rule, and the limits of linear separability.

## Useful references

- [Iris flower dataset (UCI)](https://archive.ics.uci.edu/dataset/53/iris) — the dataset we use for the perceptron.
- [Universal Approximation Theorem (paper)](https://arxiv.org/pdf/2004.08867.pdf) — why MLPs are so powerful.
- Rosenblatt, F. (1958). *The Perceptron: A probabilistic model for information storage and organization in the brain.* Psychological Review, 65(6).

---

## 1. The perceptron

A **perceptron** is the simplest possible artificial neuron. It takes a vector of inputs $\mathbf{x}$, computes a weighted sum, adds a bias, and passes the result through a *step function* to produce a binary output.

<img src="assets/perceptron.svg" alt="Single perceptron diagram" width="640"/>

Formally:

$$ z = \sum_{i=1}^{n} w_i x_i + b = \mathbf{w}^\top \mathbf{x} + b $$

$$ \hat{y} = \begin{cases} 1 & \text{if } z > 0 \\ 0 & \text{otherwise} \end{cases} $$

The **perceptron learning rule** updates weights only when the prediction is wrong:

$$ \mathbf{w} \leftarrow \mathbf{w} + \eta \, (y - \hat{y}) \, \mathbf{x} $$

where $\eta$ is the learning rate, $y$ is the true label, and $\hat{y}$ is the prediction. Notice the elegant property: if the prediction is correct, $(y - \hat{y}) = 0$ and nothing changes.

> **Key insight:** A perceptron can only learn problems that are **linearly separable** — i.e., where you can draw a single straight line (or hyperplane) between the two classes. This is the famous Minsky–Papert limitation that stalled neural network research for over a decade.

### 1.1 The dataset

We will use the **Iris flower dataset**, but restricted to two classes (*setosa* and *versicolor*) and four features. These two classes are linearly separable, so a perceptron should be able to learn the boundary.

We load the data from a local CSV file (`data/iris.csv`) rather than downloading it from the internet. This keeps the lab fully reproducible inside the Docker container even when offline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility: fix the random seed up front so your results match the notes.
RNG_SEED = 7144
np.random.seed(RNG_SEED)


def load_iris_two_class(path: str = "data/iris.csv") -> np.ndarray:
    """Load the Iris dataset and keep only the first two classes.

    Returns a (100, 5) float array where columns 0-3 are features
    (sepal length, sepal width, petal length, petal width) and column 4
    is the binary label (0 = setosa, 1 = versicolor).
    """
    df = pd.read_csv(path)
    df = df[df["class"].isin(["setosa", "versicolor"])].copy()
    df["label"] = np.where(df["class"] == "setosa", 0, 1)
    feature_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
    return df[feature_cols + ["label"]].to_numpy(dtype=np.float64)


data = load_iris_two_class()
print(f"Data shape: {data.shape}")
print(f"First three rows:\n{data[:3]}")
print(f"Class balance: {int((data[:, -1] == 0).sum())} setosa, {int((data[:, -1] == 1).sum())} versicolor")

Let's visualise the data using two of the four features. If the classes are linearly separable, we should be able to *see* a gap between them.

In [ ]:
setosa = data[data[:, -1] == 0]
versicolor = data[data[:, -1] == 1]

plt.figure(figsize=(7, 5))
plt.scatter(setosa[:, 0], setosa[:, 2], marker="o", label="setosa")
plt.scatter(versicolor[:, 0], versicolor[:, 2], marker="x", label="versicolor")
plt.xlabel("sepal length (cm)")
plt.ylabel("petal length (cm)")
plt.title("Iris: setosa vs versicolor — linearly separable")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**What to observe:** the two classes form clearly separated clouds. You could draw a straight line between them with a ruler. That is exactly what the perceptron will learn — just in 4-dimensional feature space rather than the 2D projection we see here.

### 1.2 A perceptron from scratch

We now implement the perceptron in plain NumPy. We have 4 features, so we need 4 weights, plus 1 bias term — 5 parameters in total. The bias is handled by prepending a `1` to every input vector, which lets us treat the bias as just another weight (a common trick).

The hyperparameter `num_iter` (also called *epochs*) controls how many full passes over the dataset we make. Each pass updates the weights based on every misclassified example.

In [ ]:
def perceptron(data: np.ndarray, num_iter: int, learning_rate: float = 1.0):
    """Train a single perceptron using the classic perceptron learning rule.

    Parameters
    ----------
    data : np.ndarray, shape (n_samples, n_features + 1)
        Last column is the binary label.
    num_iter : int
        Number of full passes over the dataset (epochs).
    learning_rate : float
        Step size for weight updates.

    Returns
    -------
    w : np.ndarray, shape (n_features + 1,)
        Learned weights (index 0 is the bias).
    misclassified_history : list[int]
        Number of misclassifications per epoch.
    """
    features = data[:, :-1]
    labels = data[:, -1]
    n_features = features.shape[1]

    # Initialise weights to zero (bias is w[0]).
    w = np.zeros(n_features + 1)

    misclassified_history = []

    for epoch in range(num_iter):
        misclassified = 0
        for x, label in zip(features, labels):
            # Prepend 1 so the bias is folded into the dot product.
            x_aug = np.insert(x, 0, 1.0)
            z = np.dot(w, x_aug)
            y_hat = 1.0 if z > 0 else 0.0

            error = label - y_hat
            if error != 0:
                misclassified += 1
                w = w + learning_rate * error * x_aug

        misclassified_history.append(misclassified)

    return w, misclassified_history


num_iter = 10
w, misclassified_history = perceptron(data, num_iter)

print(f"Final weights (bias first): {w}")
print(f"Misclassifications per epoch: {misclassified_history}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, num_iter + 1), misclassified_history, marker="o")
plt.xlabel("epoch")
plt.ylabel("# misclassified")
plt.title("Perceptron convergence on Iris (setosa vs versicolor)")
plt.grid(alpha=0.3)
plt.show()

**What to observe:** the misclassification count should drop to zero within a few epochs and stay there. This is the **perceptron convergence theorem** in action — if the data is linearly separable, the perceptron is *guaranteed* to find a separating hyperplane in finite time.

### 1.3 Exercise 1 — explore the perceptron

Now it's your turn. In a new code cell below, do **all three** of the following and write a short markdown answer for each:

**(a)** Re-run the perceptron with `learning_rate = 0.01`. Does it still converge in 10 epochs? Why might a smaller learning rate need more epochs?

**(b)** Modify `load_iris_two_class` to use the *versicolor vs virginica* pair instead of *setosa vs versicolor*. Train the perceptron on this new pair and plot the convergence curve. What do you observe? *(Hint: these two classes overlap in feature space.)*

**(c)** With the original setosa-vs-versicolor data, change the weight initialisation from zeros to small random values (`np.random.randn(n_features + 1) * 0.01`). Does this change the final weights? Does it change the number of epochs needed?

Use the cells below to work.

In [ ]:
# Your code for Exercise 1 (a), (b), (c) here.


*Your written observations for Exercise 1:*

(a) 

(b) 

(c) 

---

## 2. The limits of a single perceptron — and why we need MLPs

A perceptron can solve **AND**, **OR**, and **NOT** — all linearly separable. But it cannot solve **XOR**:

| x₁ | x₂ | XOR |
|----|----|-----|
| 0  | 0  | 0   |
| 0  | 1  | 1   |
| 1  | 0  | 1   |
| 1  | 1  | 0   |

Plot these four points and try to draw a single straight line separating the 1s from the 0s. You cannot — the 1s are on one diagonal and the 0s are on the other. This was Minsky and Papert's 1969 result, and it nearly killed neural network research.

The fix: **stack** perceptrons into layers, with a non-linear activation function between them. This is the **Multi-Layer Perceptron** (MLP).

<img src="assets/mlp.svg" alt="MLP architecture for XOR" width="640"/>

### 2.1 An MLP from scratch

We will build a tiny MLP with:

- **2 input neurons** (one per XOR input)
- **3 hidden neurons** with sigmoid activation
- **2 output neurons** (one-hot: `[1,0]` for class 0, `[0,1]` for class 1)

Each hidden and output neuron will also have its own **bias** term — without biases, the hidden layer's decision boundaries are forced through the origin, which makes XOR very hard or impossible to fit. Adding a learnable bias per neuron is a small change with a big effect.

We will implement forward propagation, backpropagation, and the weight update step by hand. The point is *not* to write good production code — the point is to demystify what a deep learning library does for you.

The two key non-linearities you should know:

$$ \text{sigmoid}(z) = \frac{1}{1 + e^{-z}}, \quad \text{sigmoid}'(z) = \text{sigmoid}(z) \cdot (1 - \text{sigmoid}(z)) $$

$$ \tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}, \quad \tanh'(z) = 1 - \tanh^2(z) $$

In [ ]:
# XOR dataset
XOR_data = np.array([
    [0, 0, 0],
    [0, 1, 1],
    [1, 0, 1],
    [1, 1, 0],
])
X = XOR_data[:, :2]
y = XOR_data[:, -1]

print("Inputs:")
print(X)
print("Targets:")
print(y)

In [ ]:
# --- Activation functions and their derivatives ---

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def sigmoid_derivative(activation):
    # Note: this expects the *activation* (i.e. sigmoid(z)), not z itself.
    return activation * (1.0 - activation)


def tanh(z):
    return np.tanh(z)


def tanh_derivative(activation):
    return 1.0 - activation ** 2

In [ ]:
# --- Build the network ---

def initialise_network(n_inputs: int = 2, n_hidden: int = 3, n_outputs: int = 2):
    """Create an MLP as a list of layers, each a list of neuron dicts.

    Each neuron is a dict with 'weights' and 'bias' keys. We will add 'output'
    and 'delta' keys during forward/backward passes.
    """
    rng = np.random.default_rng(RNG_SEED)
    # Xavier-style init: keeps the variance of activations roughly stable across layers.
    hidden_scale = np.sqrt(1.0 / n_inputs)
    output_scale = np.sqrt(1.0 / n_hidden)
    hidden_layer = [
        {"weights": rng.normal(0, hidden_scale, size=n_inputs), "bias": 0.0}
        for _ in range(n_hidden)
    ]
    output_layer = [
        {"weights": rng.normal(0, output_scale, size=n_hidden), "bias": 0.0}
        for _ in range(n_outputs)
    ]
    return [hidden_layer, output_layer]


def print_network(net):
    for i, layer in enumerate(net, 1):
        print(f"Layer {i} ({len(layer)} neurons)")
        for j, neuron in enumerate(layer, 1):
            print(f"  neuron {j}: weights={neuron['weights']}, bias={neuron['bias']:.3f}")


net = initialise_network()
print_network(net)

In [ ]:
# --- Forward propagation ---

def forward_propagate(net, sample):
    """Push a single input through the network and return the output layer."""
    inputs = sample
    for layer in net:
        new_inputs = []
        for neuron in layer:
            z = np.dot(neuron["weights"], inputs) + neuron["bias"]
            neuron["output"] = sigmoid(z)
            new_inputs.append(neuron["output"])
        inputs = np.array(new_inputs)
    return inputs


# --- Backpropagation: compute deltas for every neuron ---

def back_propagate(net, expected):
    """Walk the network backwards, computing the error signal (delta) for each neuron."""
    for i in reversed(range(len(net))):
        layer = net[i]
        if i == len(net) - 1:
            # Output layer: error = target - actual
            errors = [expected[j] - layer[j]["output"] for j in range(len(layer))]
        else:
            # Hidden layer: error = weighted sum of next layer's deltas
            errors = []
            for j in range(len(layer)):
                err = sum(neuron["weights"][j] * neuron["delta"] for neuron in net[i + 1])
                errors.append(err)

        for j, neuron in enumerate(layer):
            neuron["delta"] = errors[j] * sigmoid_derivative(neuron["output"])


# --- Weight update ---

def update_weights(net, sample, learning_rate):
    for i, layer in enumerate(net):
        inputs = sample if i == 0 else np.array([n["output"] for n in net[i - 1]])
        for neuron in layer:
            for j in range(len(inputs)):
                neuron["weights"][j] += learning_rate * neuron["delta"] * inputs[j]
            # Bias has an implicit input of 1.0, so its update is just lr * delta.
            neuron["bias"] += learning_rate * neuron["delta"]

In [ ]:
# --- Training loop ---

def train(net, X, y, epochs, learning_rate, n_outputs, log_every=2000):
    """Train the network and return the per-epoch sum-of-squared-errors."""
    errors_per_epoch = []
    for epoch in range(epochs):
        sum_error = 0.0
        for sample, target in zip(X, y):
            outputs = forward_propagate(net, sample)
            # one-hot encode the target
            expected = np.zeros(n_outputs)
            expected[int(target)] = 1
            sum_error += float(np.sum((expected - outputs) ** 2))
            back_propagate(net, expected)
            update_weights(net, sample, learning_rate)
        errors_per_epoch.append(sum_error)
        if epoch % log_every == 0:
            print(f"  epoch {epoch:>6d}  error = {sum_error:.4f}")
    return errors_per_epoch


# Re-initialise the network so this cell is rerunnable.
net = initialise_network()
errors = train(net, X, y, epochs=20000, learning_rate=0.5, n_outputs=2)
print(f"\nFinal error: {errors[-1]:.6f}")

**A note on epochs.** The original version of this lab used 100,000 epochs with a learning rate of 0.05 — that works but is wasteful. With a higher learning rate (0.5) and proper weight initialisation, 20,000 epochs is more than enough for XOR. A real network on a real dataset would also use *batches*, *momentum*, and *adaptive learning rates* (Adam, RMSprop) — we'll meet these later in the module.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(errors)
plt.xlabel("epoch")
plt.ylabel("sum-of-squared error")
plt.yscale("log")
plt.title("MLP learning XOR — error over training")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# --- Prediction ---

def predict(network, sample):
    """Return the predicted class index (0 or 1) for a single input."""
    outputs = forward_propagate(network, sample)
    return int(np.argmax(outputs)), outputs


print("Predictions on the four XOR inputs:")
for sample in X:
    cls, raw = predict(net, sample)
    print(f"  input={sample}  →  raw output={np.round(raw, 3)}  →  class={cls}")

### 2.2 Exercise 2 — experiment with the MLP

Do **all three** of the following:

**(a)** Re-train the MLP with only **1 hidden neuron** (call `initialise_network(n_hidden=1)`). Can it still learn XOR? Plot the error curve and explain what you see in terms of model capacity.

**(b)** Replace sigmoid with tanh everywhere in the forward and backward pass (you'll need a `tanh_derivative` variant in the appropriate places). Train for 20,000 epochs. Does tanh converge faster or slower than sigmoid on this problem? Why might that be? *(Hint: think about the gradient magnitude near zero.)*

**(c)** Set the learning rate to a much smaller value (`0.01`) and train for 20,000 epochs. Does it converge? What about `5.0`? Describe what happens at both extremes.

Use the cells below.

In [ ]:
# Your code for Exercise 2 (a), (b), (c) here.


*Your written observations for Exercise 2:*

(a) 

(b) 

(c) 

---

## 3. The high-level alternative — scikit-learn

Now we've built a perceptron and an MLP from scratch, let's see how much code disappears when we use a proper library. This is what you'd actually use in practice.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import Perceptron
from sklearn.metrics import accuracy_score

In [ ]:
# Re-create the XOR data (in case earlier variables were overwritten in your experiments).
XOR_data = np.array([[0, 0, 0], [0, 1, 1], [1, 0, 1], [1, 1, 0]])
X = XOR_data[:, :2]
y = XOR_data[:, -1]

# Sklearn's Perceptron on XOR — this should fail!
ptn = Perceptron(max_iter=500, random_state=RNG_SEED)
ptn.fit(X, y)
y_pred = ptn.predict(X)
print(f"Perceptron predictions: {y_pred}")
print(f"Perceptron accuracy on XOR: {accuracy_score(y, y_pred):.2f}")

### Why 0.5?

You should get an accuracy of **0.5** above. Looking at the predictions, the perceptron has settled on always predicting `0` — it gets 2 of the 4 XOR cases right (the two that happen to have target `0`) and 2 wrong. **This is not a bug.** It is the Minsky–Papert result expressed in code: a single linear classifier cannot solve XOR, *period*. No amount of training time, learning rate tuning, or initialisation tricks will fix it. The only solution is to add a hidden layer with a non-linear activation — which is exactly what we did with our MLP, and what `MLPClassifier` does below.

In [ ]:
# Sklearn's MLPClassifier on XOR — this should succeed.
mlp = MLPClassifier(
    hidden_layer_sizes=(3,),     # one hidden layer of 3 neurons
    activation="logistic",       # i.e. sigmoid
    solver="lbfgs",              # a good optimiser for tiny datasets
    max_iter=5000,
    random_state=RNG_SEED,
)
mlp.fit(X, y)
y_pred = mlp.predict(X)
print(f"MLP predictions: {y_pred}")
print(f"MLP accuracy on XOR: {accuracy_score(y, y_pred):.2f}")
print(f"MLP prediction for [0, 1]: {mlp.predict([[0, 1]])[0]}")

### 3.1 Exercise 3 — compare from-scratch vs library

Time your from-scratch MLP and the sklearn `MLPClassifier` on the same XOR data. Use Python's `time` module:

```python
import time
start = time.perf_counter()
# ... train ...
elapsed = time.perf_counter() - start
```

**(a)** Roughly how much faster is the library version? Why? *(Think about what sklearn is doing under the hood that your code is not.)*

**(b)** Write a short paragraph (3–5 sentences) on **when** you would write neural network code from scratch in your career, and when you would always reach for a library.

In [ ]:
# Your code for Exercise 3 (a) here.


*Your written answer for Exercise 3 (b):*



---

## 4. Reflection questions

Answer in the markdown cells below. These tie the lab back to the lecture material and will help you on the assessment. Aim for 2–4 sentences per question.

**Q1.** Explain in your own words why the perceptron learning rule only updates weights on misclassified examples. What would happen if it always updated, regardless of whether the prediction was right?

**Q2.** A colleague claims: *"A perceptron is just a single neuron, so a 'deep' network is just many perceptrons stacked together."* In what sense is this true, and in what important sense is it misleading?

**Q3.** The Universal Approximation Theorem says an MLP with one hidden layer can approximate any continuous function (given enough neurons). Why, then, do we bother building *deep* networks with many layers instead of one very wide layer?

**Q4.** In the from-scratch MLP, we used **sum-of-squared errors** as the loss. In modern classification networks you will much more often see **cross-entropy** loss. Briefly: why might cross-entropy be preferable for classification?

**Q5.** Looking at this lab as a whole, name **one** concept from today that you feel confident on, and **one** that you would like more practice with. Be specific.

*Your answers:*

**A1.** 

**A2.** 

**A3.** 

**A4.** 

**A5.** 

---

## What's next

In **Lab 2** we will move beyond toy datasets and into proper training pipelines: train/validation/test splits, batching, and using a real deep learning framework rather than rolling our own forward and backward passes.

Before leaving the container today, make sure:

- [ ] You have completed Exercises 1, 2, and 3
- [ ] You have answered the reflection questions
- [ ] Your notebook runs **top to bottom without errors** (use *Kernel → Restart and Run All*)
- [ ] You have saved your work — the `labs/` folder is volume-mounted, so your edits persist on your host machine outside the container